#### Exploratory data analysis of the API data. Objective: to better understand the data in order to design the future preprocessing pipeline.

In [94]:
import requests
import json
import pandas as pd
import time
import os

In [95]:
# Bring 100 products of the page 1

url = "https://world.openfoodfacts.org/api/v2/search"

headers = {
        "User-Agent": "Mozilla/5.0 (Data Engineering Project - student)"
    }

total_pages = 10
all_products = []

for page in range(1, total_pages + 1):
    print(f"Fetching page {page}/{total_pages}...")
    params = {
        "page_size": 100,
        "page": page
    }
    
    response = requests.get(url, params=params, headers=headers)

    if response.status_code == 200:
            data = response.json()
            products = data.get("products", [])
            all_products.extend(products)
            print(f"  → {len(products)} products fetched. Total: {len(all_products)}")
    else:
        print(f"  → Error {response.status_code} on page {page}, skipping...")
    
    time.sleep(1)  # respeta el rate limit de la API

print(f"\nTotal products fetched: {len(all_products)}")

# Save to a CSV
df = pd.DataFrame(all_products)
project_root = os.path.dirname(os.path.abspath("__file__"))
output_path = os.path.join(project_root, "data", "raw", "openfoodfacts_sample.csv")

os.makedirs(os.path.dirname(output_path), exist_ok=True)  # crea la carpeta si no existe
df.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print("Saved to CSV!")

Fetching page 1/10...
  → 100 products fetched. Total: 100
Fetching page 2/10...
  → Error 503 on page 2, skipping...
Fetching page 3/10...
  → 100 products fetched. Total: 200
Fetching page 4/10...
  → 100 products fetched. Total: 300
Fetching page 5/10...
  → 100 products fetched. Total: 400
Fetching page 6/10...
  → 100 products fetched. Total: 500
Fetching page 7/10...
  → Error 503 on page 7, skipping...
Fetching page 8/10...
  → 100 products fetched. Total: 600
Fetching page 9/10...
  → Error 503 on page 9, skipping...
Fetching page 10/10...
  → 100 products fetched. Total: 700

Total products fetched: 700
Saved to: c:\Users\34651\Desktop\python_projects\food-data-lakehouse-platform\notebooks\data\raw\openfoodfacts_sample.csv
Saved to CSV!


In [45]:
df = pd.DataFrame(products)
df.head()

,_id,_keywords,added_countries_tags,additives_n,additives_original_tags,additives_tags,allergens,allergens_from_ingredients,allergens_from_user,allergens_hierarchy,...,origin_ca,origin_et,origin_sl,packaging_text_ca,packaging_text_et,packaging_text_sl,product_name_ca,product_name_dz,product_name_et,product_name_sl
0,6111246721261,"[blanc, dessert, et, fermente, food, fromage, ...",[],1,[en:e202],[en:e202],en:milk,"en:milk, Ferment lactique",(fr) en:milk,[en:milk],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6111035000430,"[ali, and, beverage, mineral, natural, prepara...",[],0,[],[],,,(en),[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6111242100992,"[creme, dessert, fermente, jaouda, la, lacte, ...",[],0,[],[],"en:banana,en:milk","en:milk, en:milk, cream, banana","(fr) en:banana,en:milk","[en:banana, en:milk]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6111035000058,"[14001, 22000, 45001, 9001, ali, and, beverage...",[],0,[],[],,,(en),[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6111035002175,"[ali, and, bassin, beverage, cherif, dot, gree...",[],0,[],[],,,(en),[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [46]:
# Check the name of the columns
columns = df.columns

with open("columns.txt", "w", encoding="utf-8") as f:
    for item in columns:
        f.write(str(item) + "\n")

In [ ]:
# Select only important columns
FIELDS = [
    # Identification
    "code", "product_name", "generic_name", "quantity",
    "product_quantity", "product_quantity_unit",
    "serving_size", "lang", "url",
    # Brand
    "brands", "brands_tags", "owner",
    "manufacturing_places", "manufacturing_places_tags",
    # Category
    "categories", "categories_tags", "categories_hierarchy",
    "pnns_groups_1", "pnns_groups_2",
    "food_groups", "food_groups_tags",
    # Country
    "countries", "countries_tags", "countries_hierarchy",
    "origins", "origins_tags", "purchase_places",
    # Nutrition
    "nutriments", "nutrition_data_per",
    "nutriscore_grade", "nutriscore_score",
    "nova_group", "ecoscore_grade", "ecoscore_score",
    "nutrient_levels",
    # Ingredients
    "ingredients_text", "ingredients_n",
    "allergens_tags", "traces_tags",
    "additives_n", "additives_tags",
    "ingredients_analysis_tags", "labels", "labels_tags",
    # Packaging
    "packaging", "packaging_tags",
    "packaging_materials_tags", "packaging_recycling_tags",
    # Date and quality
    "created_t", "last_modified_t", "last_updated_t", "completeness",
    # Imgages
    "image_url", "image_front_url", "image_front_small_url",
]

df = df[FIELDS]

In [48]:
df.head()

,code,product_name,generic_name,quantity,product_quantity,product_quantity_unit,serving_size,serving_quantity,lang,url,...,packaging_tags,packaging_materials_tags,packaging_recycling_tags,created_t,last_modified_t,last_updated_t,completeness,image_url,image_front_url,image_front_small_url
0,6111246721261,Fromage Blanc Nature,,1 kg,1000.0,g,100 g,100.0,fr,https://world.openfoodfacts.org/product/611124...,...,[en:plastic],[en:plastic],[],1622750466,1779273710,1779273710,0.9000,https://images.openfoodfacts.org/images/produc...,https://images.openfoodfacts.org/images/produc...,https://images.openfoodfacts.org/images/produc...
1,6111035000430,Sidi Ali,,33 cl,330.0,ml,1l,1000.0,es,https://world.openfoodfacts.org/product/611103...,...,"[en:plastic, en:bottle]",[en:plastic],[],1439924914,1779352592,1779352592,0.9000,https://images.openfoodfacts.org/images/produc...,https://images.openfoodfacts.org/images/produc...,https://images.openfoodfacts.org/images/produc...
2,6111242100992,Perly,,100 g,100.0,g,80g,80.0,en,https://world.openfoodfacts.org/product/611124...,...,[en:plastic],[en:plastic],[],1474037086,1779347951,1779347951,1.0625,https://images.openfoodfacts.org/images/produc...,https://images.openfoodfacts.org/images/produc...,https://images.openfoodfacts.org/images/produc...
3,6111035000058,Eau minérale naturelle,,"1,5 L",1500.0,ml,330 ml,330.0,en,https://world.openfoodfacts.org/product/611103...,...,"[en:plastic, en:bottle-or-vial, en:bottle]",[en:plastic],[],1409671459,1779391270,1779391270,1.0000,https://images.openfoodfacts.org/images/produc...,https://images.openfoodfacts.org/images/produc...,https://images.openfoodfacts.org/images/produc...
4,6111035002175,Sidi Ali,,2 L,2000.0,ml,NaN,NaN,fr,https://world.openfoodfacts.org/product/611103...,...,[],[en:plastic],[],1537111522,1775998974,1775998974,0.8000,https://images.openfoodfacts.org/images/produc...,https://images.openfoodfacts.org/images/produc...,https://images.openfoodfacts.org/images/produc...


In [51]:
# Analize the completeness column that indicates if the products has lot of empty columns or not
print(min(df["completeness"]))
print(max(df["completeness"]))

0.55
1.1


In [ ]:
# Lets analize numeric columns
df.describe()


,product_quantity,serving_quantity,nutriscore_score,nova_group,ecoscore_score,additives_n,created_t,last_modified_t,last_updated_t,completeness
count,98.000000,66.000000,98.000000,85.000000,76.000000,100.000000,1.000000e+02,1.000000e+02,1.000000e+02,100.000000
mean,579.683673,210.398485,6.795918,3.070588,55.855263,0.870000,1.518673e+09,1.778652e+09,1.778824e+09,0.869500
std,755.180290,331.768456,10.002020,1.232383,21.536762,1.454043,1.142327e+08,2.911931e+06,1.709576e+06,0.140046
min,22.000000,1.000000,-10.000000,1.000000,12.000000,0.000000,1.337517e+09,1.754122e+09,1.771266e+09,0.550000
25%,202.500000,25.425000,0.000000,3.000000,38.000000,0.000000,1.439157e+09,1.779348e+09,1.779348e+09,0.787500
50%,350.000000,50.000000,2.000000,4.000000,57.000000,0.000000,1.505096e+09,1.779419e+09,1.779419e+09,0.900000
75%,637.500000,145.000000,12.750000,4.000000,71.000000,1.000000,1.605782e+09,1.779496e+09,1.779496e+09,1.000000
max,5000.000000,1000.000000,31.000000,4.000000,97.000000,9.000000,1.741827e+09,1.779702e+09,1.779702e+09,1.100000


In [ ]:
# It cannot exist rows with completeness > 1, because it means than more than 100% of the data is filled, and that has no sense
# We will delete rows with column "completeness" > 1.
df[df["completeness"] > 1].shape[0]

8

In [61]:
df[["created_t", "last_modified_t", "last_updated_t"]].head()

,created_t,last_modified_t,last_updated_t
0,1622750466,1779273710,1779273710
1,1439924914,1779352592,1779352592
2,1474037086,1779347951,1779347951
3,1409671459,1779391270,1779391270
4,1537111522,1775998974,1775998974


In [73]:
analized_columns = ["created_t", "last_modified_t", "last_updated_t", "completeness", "serving_quantity",
                    "product_quantity", "serving_quantity", "nutriscore_score", "nova_group",
                    "ecoscore_score", "additives_n"]

In [77]:
df_other_columns = df.drop(columns=analized_columns, errors="ignore")

In [84]:
print(df_other_columns["nutriments"][0])

{'added-sugars': 0, 'added-sugars_100g': 0, 'added-sugars_serving': 0, 'added-sugars_unit': 'g', 'added-sugars_value': 0, 'alcohol': 0, 'alcohol_100g': 0, 'alcohol_serving': 0, 'alcohol_unit': '% vol', 'alcohol_value': 0, 'carbohydrates': 10, 'carbohydrates_100g': 10, 'carbohydrates_serving': 10, 'carbohydrates_unit': 'g', 'carbohydrates_value': 10, 'energy': 643, 'energy-kcal': 159, 'energy-kcal_100g': 159, 'energy-kcal_serving': 159, 'energy-kcal_unit': 'kcal', 'energy-kcal_value': 159, 'energy-kj': 643, 'energy-kj_100g': 643, 'energy-kj_serving': 643, 'energy-kj_unit': 'kJ', 'energy-kj_value': 643, 'energy_100g': 643, 'energy_serving': 643, 'energy_unit': 'kJ', 'energy_value': 643, 'fat': 11, 'fat_100g': 11, 'fat_serving': 11, 'fat_unit': 'g', 'fat_value': 11, 'fiber': 0, 'fiber_100g': 0, 'fiber_serving': 0, 'fiber_unit': 'g', 'fiber_value': 0, 'fruits-vegetables-legumes-estimate-from-ingredients_100g': 0, 'fruits-vegetables-nuts-estimate-from-ingredients_100g': 0, 'nova-group': 3, 

In [64]:
#	product_quantity	serving_quantity	nutriscore_score	nova_group	ecoscore_score	additives_n

df["product_quantity"]

0     1000.0
1      330.0
2      100.0
3     1500.0
4     2000.0
       ...  
95     100.0
96     125.0
97     750.0
98     280.0
99     360.0
Name: product_quantity, Length: 100, dtype: float64

In [71]:
print(min(df["product_quantity"]))
print(max(df["product_quantity"]))
print(df["product_quantity"].isna().sum())

22.0
5000.0
2


#### Data quality Issues
##### 1. Delete rows with column "completeness" > 1. 
##### 2. Change the format of the following columns: created_t, last_modified_t, last_updated_t. Transform from Timestamp in Unix format into datetime staging layer 
##### 3. Remove rows where df["last_modified_t"] < df["created_t"] because it does not make sense to update products before been created
##### 4. Remove rows with column "code" null because is the PK

#### Other Data Quality Issue that will not be corrected
##### 1.- Null rows in ["product_quantity"	"serving_quantity"	"nutriscore_score"	"nova_group"	"ecoscore_score"] columns.